# Feature Engineering for Ames Housing

This notebook covers:
1. Creating new features from existing ones
2. Handling categorical variables
3. Feature transformation
4. Feature selection

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Load cleaned data
df = pd.read_csv('../data/raw/AmesHousing.csv')
print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (2930, 82)


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


## 1. Create New Features

Based on domain knowledge about housing prices

In [17]:
def create_features(df):
    """Create new features from existing data."""
    df_new = df.copy()
    
    # 1. Age features
    df_new['Age'] = df_new['Yr Sold'] - df_new['Year Built']
    df_new['Age_At_Remodel'] = df_new['Year Remod/Add'] - df_new['Year Built']
    df_new['Is_Remodeled'] = (df_new['Age_At_Remodel'] > 0).astype(int)
    df_new['Age_Group'] = pd.cut(df_new['Age'], bins=[0, 10, 25, 50, 100, 200], 
                                 labels=['New', 'Modern', 'Mid', 'Old', 'Very Old'])
    
    # 2. Area features
    df_new['Total_SF'] = df_new['1st Flr SF'] + df_new['2nd Flr SF']
    df_new['Total_Basement_SF'] = df_new['Total Bsmt SF']
    df_new['Total_Area'] = df_new['Total_SF'] + df_new['Total_Basement_SF']
    df_new['Lot_Area_Acres'] = df_new['Lot Area'] / 43560  # Convert to acres
    df_new['SF_per_Lot'] = df_new['Total_Area'] / (df_new['Lot Area'] + 1)
    
    # 3. Room features
    df_new['Total_Bathrooms'] = (df_new['Bsmt Full Bath'] + 
                                df_new['Bsmt Half Bath'] * 0.5 + 
                                df_new['Full Bath'] + 
                                df_new['Half Bath'] * 0.5)
    df_new['Total_Rooms'] = df_new['TotRms AbvGrd'] + df_new['Bedroom AbvGr']
    df_new['Bedroom_Per_Room'] = df_new['Bedroom AbvGr'] / (df_new['TotRms AbvGrd'] + 1)
    df_new['Bathroom_Per_Room'] = df_new['Total_Bathrooms'] / (df_new['TotRms AbvGrd'] + 1)
    df_new['Has_2nd_Floor'] = (df_new['2nd Flr SF'] > 0).astype(int)
    df_new['Has_Basement'] = (df_new['Total_Basement_SF'] > 0).astype(int)
    
    # 4. Porch/Outdoor features
    df_new['Total_Porch_SF'] = (df_new['Wood Deck SF'] + 
                               df_new['Open Porch SF'] + 
                               df_new['Enclosed Porch'] + 
                               df_new['3Ssn Porch'] + 
                               df_new['Screen Porch'])
    df_new['Has_Porch'] = (df_new['Total_Porch_SF'] > 0).astype(int)
    df_new['Has_Pool'] = (df_new['Pool Area'] > 0).astype(int)
    df_new['Has_Fireplace'] = (df_new['Fireplaces'] > 0).astype(int)
    
    # 5. Garage features
    df_new['Has_Garage'] = (df_new['Garage Area'] > 0).astype(int)
    df_new['Garage_Cars_Per_Area'] = df_new['Garage Cars'] / (df_new['Garage Area'] + 1) * 100
    df_new['Garage_Quality'] = df_new['Garage Qual'].map({'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'NA': 0})
    
    # 6. Quality features
    df_new['Quality_Score'] = df_new['Overall Qual'] * df_new['Overall Cond']
    df_new['Quality_Per_Room'] = df_new['Overall Qual'] / (df_new['TotRms AbvGrd'] + 1)
    df_new['Exterior_Quality'] = (df_new['Exter Qual'].map({'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1})
                                 + df_new['Exter Cond'].map({'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1}))
    
    # 7. Sale features
    df_new['Is_New_Construction'] = (df_new['Sale Condition'] == 'New').astype(int)
    df_new['Is_Family_Sale'] = (df_new['Sale Condition'] == 'Family').astype(int)
    df_new['Is_Abnormal_Sale'] = (df_new['Sale Condition'].isin(['Abnorml', 'AdjLand', 'Alloca'])).astype(int)
    df_new['Sale_Year_Month'] = df_new['Yr Sold'] * 12 + df_new['Mo Sold']
    
    # 8. Neighborhood features (simple encoding)
    neighborhood_price = df_new.groupby('Neighborhood')['SalePrice'].median()
    df_new['Neighborhood_Price_Level'] = df_new['Neighborhood'].map(neighborhood_price)
    df_new['Neighborhood_Price_Level_Rank'] = df_new['Neighborhood_Price_Level'].rank(pct=True)
    
    return df_new

# Create features
df_featured = create_features(df)
print(f'New shape: {df_featured.shape}')
print(f'New features created: {df_featured.shape[1] - df.shape[1]}')

New shape: (2930, 113)
New features created: 31


## 2. Handle Categorical Variables

In [18]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Identify categorical columns
categorical_cols = df_featured.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical columns: {len(categorical_cols)}')
print(categorical_cols[:10])

# Encode categorical variables
df_encoded = df_featured.copy()
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    label_encoders[col] = le

print(f'Encoded shape: {df_encoded.shape}')

Categorical columns: 43
['MS Zoning', 'Street', 'Alley', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1']


Encoded shape: (2930, 113)


## 3. Feature Transformation

In [19]:
from sklearn.preprocessing import StandardScaler, PowerTransformer

# Log transform skewed features
numeric_cols = df_encoded.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('SalePrice')

# Check skewness
skewness = df_encoded[numeric_cols].skew().sort_values(ascending=False)
highly_skewed = skewness[skewness > 1].index.tolist()
print(f'Highly skewed features ({len(highly_skewed)}): {highly_skewed[:10]}')

# Apply log transformation to highly skewed features
df_transformed = df_encoded.copy()
for col in highly_skewed:
    if (df_transformed[col] > 0).all():
        df_transformed[col] = np.log1p(df_transformed[col])
    else:
        # Use Yeo-Johnson for features with negative values
        pt = PowerTransformer(method='yeo-johnson')
        df_transformed[col] = pt.fit_transform(df_transformed[[col]])

print('Features transformed')

Highly skewed features (38): ['Utilities', 'Misc Val', 'Pool Area', 'Has_Pool', 'Lot_Area_Acres', 'Lot Area', 'Low Qual Fin SF', 'Heating', 'Condition 2', '3Ssn Porch']


Features transformed


## 4. Feature Selection

In [20]:
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.model_selection import train_test_split

# Split data
X = df_transformed.drop('SalePrice', axis=1)
y = df_transformed['SalePrice']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature selection using mutual information
selector = SelectKBest(score_func=mutual_info_regression, k=50)
X_selected = selector.fit_transform(X_train, y_train)

# Get selected feature names
selected_mask = selector.get_support()
selected_features = X_train.columns[selected_mask].tolist()

print(f'Selected {len(selected_features)} features out of {len(X_train.columns)}')
print('\nTop 20 selected features:')
feature_scores = pd.DataFrame({
    'Feature': X_train.columns,
    'Score': selector.scores_
}).sort_values('Score', ascending=False)
print(feature_scores.head(20))

ValueError: Input contains NaN

## 5. Correlation Analysis with Target

In [ ]:
# Correlation with target
correlations = df_transformed.corr()['SalePrice'].sort_values(ascending=False)

plt.figure(figsize=(12, 8))
correlations.head(30).plot(kind='bar')
plt.title('Top 30 Features Correlated with SalePrice')
plt.xlabel('Features')
plt.ylabel('Correlation')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Save Processed Data

In [ ]:
# Save processed data
df_transformed.to_csv('../data/processed/ames_processed.csv', index=False)
print('Processed data saved to data/processed/ames_processed.csv')

# Save feature importance for later use
feature_scores.to_csv('../models/feature_importance.csv', index=False)
print('Feature importance saved to models/feature_importance.csv')